# Gráfico da loss de treino

Este notebook reporta a curva da **loss de treino**.

Ele lê `train_loss.csv` no diretório da rodada em `previsoes/`. Se o CSV ainda não existir, tenta extrair a loss dos logs em `logs/`, buscando linhas no formato `Epoch X/Y | Train loss: Z`.


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    ROOT_DIR = Path(__file__).resolve().parents[1]
except NameError:
    cwd = Path.cwd()
    ROOT_DIR = cwd.parent if cwd.name == "utils" else cwd

# Ajuste este caminho para a rodada que deseja avaliar.
PREVISOES_DIR = ROOT_DIR / "previsoes" / "b3_daily_financeiro" / "AttentionSoloNaive" / "2026-05-27"

# Arquivo gerado pelo main_test.py ao fim do treino.
TRAIN_LOSS_PATH = PREVISOES_DIR / "train_loss.csv"

# Fallback: logs SLURM ou logs comuns contendo "Epoch X/Y | Train loss: Z".
LOGS_DIR = ROOT_DIR / "logs"


def carregar_train_loss(train_loss_path: Path = None, logs_dir: Path = None) -> pd.DataFrame:
    """
    Carrega loss de treino de:
      1. train_loss.csv com colunas epoch, train_loss; ou
      2. logs com linhas: Epoch X/Y | Train loss: Z
    """
    train_loss_path = Path(train_loss_path) if train_loss_path else None

    if train_loss_path and train_loss_path.exists():
        df = pd.read_csv(train_loss_path)

        loss_col = next(
            (col for col in ["train_loss", "loss", "Loss", "TRAIN_LOSS"] if col in df.columns),
            None,
        )
        if loss_col is None:
            raise ValueError(
                f"{train_loss_path} não possui coluna de loss. "
                "Use 'train_loss' ou 'loss'."
            )

        epoch_col = "epoch" if "epoch" in df.columns else None
        out = pd.DataFrame({
            "epoch": df[epoch_col] if epoch_col else np.arange(1, len(df) + 1),
            "train_loss": df[loss_col],
        })
        out["source"] = train_loss_path.name
        return out

    logs_dir = Path(logs_dir) if logs_dir else None
    if logs_dir is None or not logs_dir.exists():
        return pd.DataFrame(columns=["epoch", "train_loss", "source"])

    padrao = re.compile(
        r"Epoch\s+(\d+)\s*/\s*(\d+)\s*\|\s*Train\s+loss:\s*([0-9.+\-eE]+)"
    )

    registros = []
    arquivos_log = sorted(
        list(logs_dir.glob("*.out")) + list(logs_dir.glob("*.log")) + list(logs_dir.glob("*.txt")),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )

    for log_file in arquivos_log:
        texto = log_file.read_text(encoding="utf-8", errors="ignore")
        for match in padrao.finditer(texto):
            registros.append({
                "epoch": int(match.group(1)),
                "total_epochs": int(match.group(2)),
                "train_loss": float(match.group(3)),
                "source": log_file.name,
            })

    if not registros:
        return pd.DataFrame(columns=["epoch", "train_loss", "source"])

    return pd.DataFrame(registros).sort_values(["source", "epoch"])


def plot_train_loss(train_loss_path: Path = None, logs_dir: Path = None, save_path: Path = None):
    """Reporta e plota a loss de treino."""
    loss_df = carregar_train_loss(train_loss_path=train_loss_path, logs_dir=logs_dir)

    if loss_df.empty:
        print("⚠️ Loss de treino não encontrada.")
        print(f"Procurei por CSV em: {train_loss_path}")
        print(f"Procurei por logs em: {logs_dir}")
        print("Esperado: train_loss.csv ou logs com 'Epoch X/Y | Train loss: Z'.")
        return loss_df

    plt.figure(figsize=(10, 4))

    if loss_df["source"].nunique() > 1:
        for source, d in loss_df.groupby("source"):
            d = d.sort_values("epoch")
            plt.plot(d["epoch"], d["train_loss"], marker="o", linewidth=1.5, label=source)
        plt.legend()
    else:
        d = loss_df.sort_values("epoch")
        plt.plot(d["epoch"], d["train_loss"], marker="o", linewidth=1.8)

    plt.title("Loss de treino por época")
    plt.xlabel("Época")
    plt.ylabel("Train loss")
    plt.grid(alpha=0.3)
    plt.tight_layout()

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"✅ Gráfico salvo em: {save_path}")

    plt.show()
    return loss_df


loss_df = plot_train_loss(
    train_loss_path=TRAIN_LOSS_PATH,
    logs_dir=LOGS_DIR,
    save_path=PREVISOES_DIR / "train_loss_report.png",
)

display(loss_df.head(20))
